# TCD to TCN 변환

> 교통카드 데이터(TCD)를 통행체인 네트워크(TCN) 형식으로 변환하는 파이프라인

**목적**: 원시 TCD 데이터를 정제하여 유효한 통행 기록(TCN)으로 변환
- 동일 O-D 제거
- 좌표 누락 데이터 제거  
- 500m 미만 단거리 통행 필터링
- parquet 형식으로 저장

---
## 1. 설정

In [108]:
%load_ext autoreload
%autoreload

from module.tcd_to_tcn_route import process_multiple_dates

import warnings

warnings.filterwarnings('ignore')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


---
## 2. 데이터 변환 실행

In [109]:
# dates = ['20250217', '20250218', '20250219', '20250220', '20250221', '20250222', '20250223']
dates = ['20250217']

results = process_multiple_dates(
    dates,
    base_path='C:/Folder/Research/0. DATA/tcd_2025_parquet',
    output_dir='../data/tcn',
    split_round_trip=False,
    sig_path='../data/shp/sig',
    ctprvn_path='../data/shp/ctprvn',
)

Processing 20250217 (route-based)...
  Loaded TCD: 17,826,894 records
  Loaded ROUT: 4,010 routes
  Loaded ROUTESTTN: 257,579 route-station records
  After removing same O-D: 17,430,369 records
  After removing NA station ID: 17,213,257 records (removed 187,318 trips)
  Route-station match: origin 17,138,646/17,213,257 (99.6%), dest 17,121,036/17,213,257 (99.5%)
  After removing zero coords: 17,183,306 records (removed 19,097 trips)
  After removing NA coords: 16,911,748 records (removed 154,356 trips)
  SubwayTransferGraph: 24 lines, 109 transfer stations
  Origin coords fixed: 0
  Destination coords fixed: 0
  Before distance filter: 13,188,341
  After distance filter (>= 500m): 12,999,778
  Removed short trips: 188,563
  Final TCN: 12,999,778 trips
  After region filter: 11,981,768
  Saved to ../data/tcn\20250217\TCN_20250217_route.parquet


---
## 3. 결과 확인

In [ ]:
# 결과 확인
for date, tcn in results.items():
    hidden = (tcn['숨겨진환승횟수'] > 0).sum()
    print(f'{date}: {len(tcn):,} trips, 숨겨진환승 {hidden:,}건 ({hidden/len(tcn)*100:.1f}%)')

---
## 4. Route 기반 변환 테스트 (tcd_to_tcn_route)

> STTN 대신 ROUTE/ROUTESTTN 기반으로 정류장을 매칭하는 새 방식 테스트  
> TCD 1000건 샘플로 단계별 실행

In [ ]:
%load_ext autoreload
%autoreload

from module.tcd_to_tcn_route import TCDLoader, TCDPreprocessor, TCDtoTCNConverter, SubwayTransferGraph

In [82]:
# 4-1. 데이터 로드
date = '20250217'
loader = TCDLoader(base_path=r'C:\Folder\Research\0. DATA\tcd_2025_parquet')
tcd_raw, route, routesttn = loader.load_data(date)

print(f'TCD: {len(tcd_raw):,}건')
print(f'ROUTE: {len(route):,}건')
print(f'ROUTESTTN: {len(routesttn):,}건')

TCD: 17,826,894건
ROUTE: 4,010건
ROUTESTTN: 257,579건


In [83]:
# 4-2. TCD 1000건 샘플링 (통행 단위로 추출)
unique_trips = tcd_raw[['가상카드번호', '트랜잭션ID']].drop_duplicates()
sampled_trips = unique_trips.head(1000)
tcd_sample = tcd_raw.merge(sampled_trips, on=['가상카드번호', '트랜잭션ID'], how='inner')

print(f'샘플 통행 수: {len(sampled_trips):,}건')
print(f'샘플 TCD 레코드: {len(tcd_sample):,}건')
tcd_sample.head()

샘플 통행 수: 1,000건
샘플 TCD 레코드: 1,004건


,운행일자,정산사 ID,인련번호,가상카드번호,정산지역코드,카드구분코드,차량ID(국토부표준),차량ID(정산사업자),차량등록번호,운행출발일시,...,승차정류장ID(정산사업자),하차정류장ID(국토부표준),하차정류장ID(정산사업자),하차일시,트랜잭션ID,환승건수,사용자구분코드,이용자수,이용거리,탑승시간
0,20250217,3,1,100031869547,MM10144000,C,NaN,129721033.0,충남72자1033,2.025022e+13,...,2921216,NaN,2921208.0,2.025022e+13,1,0,1,1,2333,420
1,20250217,3,2,100031869547,MM10144000,C,NaN,129721038.0,충남72자1038,2.025022e+13,...,2921234,NaN,2921220.0,2.025022e+13,2,0,1,1,4366,593
2,20250217,3,1,100032682616,MM10144000,C,NaN,129701314.0,충남70자1314,2.025022e+13,...,2921412,NaN,2921392.0,2.025022e+13,1,0,1,1,21914,2261
3,20250217,3,2,100032682616,MM10144000,C,NaN,129701247.0,충남70자1247,2.025022e+13,...,2921394,NaN,2923272.0,2.025022e+13,2,0,1,1,21619,2733
4,20250217,3,1,100033352632,MM10144000,C,NaN,129721111.0,충남72자1111,2.025022e+13,...,2921214,NaN,2921208.0,2.025022e+13,1,0,1,1,1516,206


In [84]:
# 4-3. 승하차 동일 정류장 제거
mask_same = tcd_sample['승차정류장ID(정산사업자)'] == tcd_sample['하차정류장ID(정산사업자)']
invalid = tcd_sample.loc[mask_same, ['가상카드번호', '트랜잭션ID']].drop_duplicates()
tcd_clean = tcd_sample.merge(
    invalid, on=['가상카드번호', '트랜잭션ID'], how='left', indicator=True
).query("_merge == 'left_only'").drop(columns='_merge')

print(f'동일 O-D 제거: {len(tcd_sample):,} → {len(tcd_clean):,} ({len(tcd_sample)-len(tcd_clean):,}건 제거)')

동일 O-D 제거: 1,004 → 978 (26건 제거)


In [85]:
# 4-3.5 정류장ID가 NA인 통행 제거 (승차 또는 하차 정류장ID가 NA면 해당 통행 전체 제거)
sttn_cols = ['승차정류장ID(정산사업자)', '하차정류장ID(정산사업자)']
mask_na_sttn = tcd_clean[sttn_cols].isna().any(axis=1)
invalid_na_sttn = tcd_clean.loc[mask_na_sttn, ['가상카드번호', '트랜잭션ID']].drop_duplicates()

tcd_clean = tcd_clean.merge(
    invalid_na_sttn, on=['가상카드번호', '트랜잭션ID'], how='left', indicator=True
).query("_merge == 'left_only'").drop(columns='_merge')

print(f'정류장ID NA 통행 제거: {len(invalid_na_sttn):,}건 통행 제거')
print(f'남은 레코드: {len(tcd_clean):,}건')

정류장ID NA 통행 제거: 54건 통행 제거
남은 레코드: 924건


In [86]:
# 4-4. TCD 전처리
preprocessor = TCDPreprocessor()
tcd_pre = preprocessor.preprocess_tcd(tcd_clean)

print(f'전처리 후 컬럼: {tcd_pre.columns.tolist()}')
tcd_pre.head()

전처리 후 컬럼: ['운행일자', '정산사 ID', '인련번호', '가상카드번호', '정산지역코드', '카드구분코드', '차량ID(정산사업자)', '차량등록번호', '운행출발일시', '운행종료일시', '교통수단코드', '노선ID', '승차일시', '발권일시', '승차정류장ID', '하차정류장ID', '하차일시', '트랜잭션ID', '환승건수', '사용자구분코드', '이용자수', '이용거리', '탑승시간']


,운행일자,정산사 ID,인련번호,가상카드번호,정산지역코드,카드구분코드,차량ID(정산사업자),차량등록번호,운행출발일시,운행종료일시,...,발권일시,승차정류장ID,하차정류장ID,하차일시,트랜잭션ID,환승건수,사용자구분코드,이용자수,이용거리,탑승시간
0,20250217,3,1,100031869547,MM10144000,C,129721033.0,충남72자1033,2.025022e+13,NaN,...,NaN,2921216,2921208,20250217081222,1,0,1,1,2333,420
1,20250217,3,2,100031869547,MM10144000,C,129721038.0,충남72자1038,2.025022e+13,NaN,...,NaN,2921234,2921220,20250217170221,2,0,1,1,4366,593
2,20250217,3,1,100032682616,MM10144000,C,129701314.0,충남70자1314,2.025022e+13,NaN,...,NaN,2921412,2921392,20250217085158,1,0,1,1,21914,2261
3,20250217,3,2,100032682616,MM10144000,C,129701247.0,충남70자1247,2.025022e+13,NaN,...,NaN,2921394,2923272,20250217134002,2,0,1,1,21619,2733
4,20250217,3,1,100033352632,MM10144000,C,129721111.0,충남72자1111,2.025022e+13,NaN,...,NaN,2921214,2921208,20250217172035,1,0,1,1,1516,206


In [87]:
# 4-5. ROUTESTTN 전처리
routesttn_pre = preprocessor.preprocess_routesttn(routesttn)

print(f'ROUTESTTN 전처리 후: {len(routesttn_pre):,}건')
print(f'컬럼: {routesttn_pre.columns.tolist()}')
routesttn_pre.head()

ROUTESTTN 전처리 후: 253,755건
컬럼: ['운행일자', '정산사 ID', '정산지역코드', '노선ID', '노선명(short)', '교통수단유형', '정류장순서', '정류장 ID', '정류장 명칭', '정류장 X 좌표', '정류장 Y 좌표', '누적거리(m)', '구간거리(m)']


,운행일자,정산사 ID,정산지역코드,노선ID,노선명(short),교통수단유형,정류장순서,정류장 ID,정류장 명칭,정류장 X 좌표,정류장 Y 좌표,누적거리(m),구간거리(m)
0,20250217,3,MM10142000,43703201,57(상구현)도계삼산교,B,0,4372031,관설동종점,37.30070,127.98265,0,0
1,20250217,3,MM10142000,43703201,57(상구현)도계삼산교,B,1,4370031,원주자동차운전학원,37.30228,127.98100,228,228
2,20250217,3,MM10142000,43703201,57(상구현)도계삼산교,B,2,4388631,학마을,37.30463,127.97937,540,312
3,20250217,3,MM10142000,43703201,57(상구현)도계삼산교,B,3,4388581,당둔지,37.30737,127.97856,860,320
4,20250217,3,MM10142000,43703201,57(상구현)도계삼산교,B,4,4372281,영서고등학교,37.30994,127.97662,1198,338


In [88]:
# 4-6. TCD + ROUTESTTN 병합 (노선ID + 정류장ID 기반)
tcd_merged = preprocessor.merge_tcd_routesttn(tcd_pre, routesttn_pre)
tcd_merged = preprocessor.create_trip_id(tcd_merged)

# 매칭 현황
o_matched = tcd_merged['승차정류장 X 좌표'].notna().sum()
d_matched = tcd_merged['하차정류장 X 좌표'].notna().sum()
total = len(tcd_merged)
print(f'총 레코드: {total:,}건')
print(f'승차 좌표 매칭: {o_matched:,}/{total:,} ({o_matched/total*100:.1f}%)')
print(f'하차 좌표 매칭: {d_matched:,}/{total:,} ({d_matched/total*100:.1f}%)')
print(f'\n병합 후 컬럼: {tcd_merged.columns.tolist()}')
tcd_merged.head()

총 레코드: 924건
승차 좌표 매칭: 924/924 (100.0%)
하차 좌표 매칭: 924/924 (100.0%)

병합 후 컬럼: ['운행일자', '정산사 ID', '인련번호', '가상카드번호', '정산지역코드', '카드구분코드', '차량ID(정산사업자)', '차량등록번호', '운행출발일시', '운행종료일시', '교통수단코드', '노선ID', '승차일시', '발권일시', '승차정류장ID', '하차정류장ID', '하차일시', '트랜잭션ID', '환승건수', '사용자구분코드', '이용자수', '이용거리', '탑승시간', '승차정류장 명칭', '승차정류장 X 좌표', '승차정류장 Y 좌표', '승차교통수단유형', '승차노선명', '승차정류장순서', '승차누적거리', '하차정류장 명칭', '하차정류장 X 좌표', '하차정류장 Y 좌표', '하차교통수단유형', '하차노선명', '하차정류장순서', '하차누적거리', '환승횟수_재계산', '구분코드']


,운행일자,정산사 ID,인련번호,가상카드번호,정산지역코드,카드구분코드,차량ID(정산사업자),차량등록번호,운행출발일시,운행종료일시,...,승차누적거리,하차정류장 명칭,하차정류장 X 좌표,하차정류장 Y 좌표,하차교통수단유형,하차노선명,하차정류장순서,하차누적거리,환승횟수_재계산,구분코드
0,20250217,3,1,100031869547,MM10144000,C,129721033.0,충남72자1033,2.025022e+13,NaN,...,36847,사거리슈퍼,36.94473,127.05297,B,531,78.0,38912.0,0,100031869547_1
1,20250217,3,2,100031869547,MM10144000,C,129721038.0,충남72자1038,2.025022e+13,NaN,...,19615,팽성전화국,36.96023,127.05868,B,500,39.0,23606.0,0,100031869547_2
2,20250217,3,1,100032682616,MM10144000,C,129701314.0,충남70자1314,2.025022e+13,NaN,...,29835,단국대학교병원,36.83940,127.17425,B,201,104.0,51315.0,0,100032682616_1
3,20250217,3,2,100032682616,MM10144000,C,129701247.0,충남70자1247,2.025022e+13,NaN,...,6903,인지사거리,37.00440,127.26925,B,201,57.0,28008.0,0,100032682616_2
4,20250217,3,1,100033352632,MM10144000,C,129721111.0,충남72자1111,2.025022e+13,NaN,...,37020,사거리슈퍼,36.94473,127.05297,B,500,73.0,37797.0,0,100033352632_1


1. 환승 1호선 -. 2호선으로 가는 환승 존재(중간 환승이 없음) 이게 한 행에서도 나타남( 찍고 간게 아니라서)
15. 1호선 -. 겅의 중앙선
- 지하철일때 - routesttn도 지하철 정류장으로 붙이기

2. route의 노선 id 는 002 인데 routesttn에서는 2임

In [95]:
tcd_merged.iloc[116:117]

,운행일자,정산사 ID,인련번호,가상카드번호,정산지역코드,카드구분코드,차량ID(정산사업자),차량등록번호,운행출발일시,운행종료일시,...,승차누적거리,하차정류장 명칭,하차정류장 X 좌표,하차정류장 Y 좌표,하차교통수단유형,하차노선명,하차정류장순서,하차누적거리,환승횟수_재계산,구분코드
116,20250217,8,1,D00230E12B897EAA,11100,4,NaN,None,NaN,NaN,...,0,역삼,37.500661,127.03643,T,2호선,21.0,0.0,0,D00230E12B897EAA_94


In [ ]:
# 4-7. 좌표 0 / NA인 통행 제거
coord_cols = ['승차정류장 X 좌표', '승차정류장 Y 좌표', '하차정류장 X 좌표', '하차정류장 Y 좌표']

# 좌표 0인 통행 제거
mask_zero = (tcd_merged[coord_cols] == 0).any(axis=1)
invalid_zero = tcd_merged.loc[mask_zero, ['가상카드번호', '트랜잭션ID']].drop_duplicates()

tcd_merged = tcd_merged.merge(
    invalid_zero, on=['가상카드번호', '트랜잭션ID'], how='left', indicator=True
).query("_merge == 'left_only'").drop(columns='_merge')
print(f'좌표 0 통행 제거: {len(invalid_zero):,}건 통행 제거 → 남은 레코드: {len(tcd_merged):,}건')

# 좌표 NA인 통행 제거
mask_na = tcd_merged[coord_cols].isna().any(axis=1)
invalid_na = tcd_merged.loc[mask_na, ['가상카드번호', '트랜잭션ID']].drop_duplicates()

tcd_final = tcd_merged.merge(
    invalid_na, on=['가상카드번호', '트랜잭션ID'], how='left', indicator=True
).query("_merge == 'left_only'").drop(columns='_merge')
print(f'좌표 NA 통행 제거: {len(invalid_na):,}건 통행 제거 → 남은 레코드: {len(tcd_final):,}건')

In [ ]:
# 4-8. TCN 변환 (숨겨진 환승 그래프 포함)
transfer_graph = SubwayTransferGraph(routesttn)
converter = TCDtoTCNConverter(split_round_trip=True, transfer_graph=transfer_graph)
tcn = converter.convert(tcd_final)
tcn = tcn.dropna().reset_index(drop=True)

print(f'최종 TCN: {len(tcn):,} 통행')
print(f'컬럼: {tcn.columns.tolist()}')
tcn.head()

In [ ]:
# 4-9. 결과 상세 확인
print('=== 교통수단유형 분포 ===')
if '교통수단유형' in tcn.columns:
    from collections import Counter
    all_modes = [m for modes in tcn['교통수단유형'] for m in modes]
    print(Counter(all_modes))

print(f'\n=== 환승횟수 분포 ===')
print(tcn['환승횟수'].value_counts().sort_index())

print(f'\n=== 숨겨진환승 있는 통행 ===')
hidden = tcn[tcn['숨겨진환승횟수'] > 0]
print(f'숨겨진환승 통행 수: {len(hidden):,} / {len(tcn):,} ({len(hidden)/len(tcn)*100:.1f}%)')

if len(hidden) > 0:
    print(f'\n=== 숨겨진환승 예시 (상위 5건) ===')
    show_cols = ['승차정류장 명칭', '하차정류장 명칭', '정류장명칭시퀀스', '숨겨진환승역', 
                 '명시적환승횟수', '숨겨진환승횟수', '환승횟수']
    show_cols = [c for c in show_cols if c in hidden.columns]
    display(hidden[show_cols].head())

print(f'\n=== 정류장명칭시퀀스 예시 (상위 5건) ===')
show_cols2 = ['승차정류장 명칭', '하차정류장 명칭', '정류장명칭시퀀스', '환승횟수']
show_cols2 = [c for c in show_cols2 if c in tcn.columns]
display(tcn[show_cols2].head())

---
## 5. Route 기반 전체 변환 실행

In [ ]:
%load_ext autoreload
%autoreload

from module.tcd_to_tcn_route import process_multiple_dates

import warnings
warnings.filterwarnings('ignore')

dates = ['20250217', '20250218', '20250219', '20250220', '20250221', '20250222', '20250223']

results = process_multiple_dates(
    dates,
    base_path=r'C:\Folder\Research\0. DATA\tcd_2025_parquet',
    output_dir='../data/tcn_route',
    split_round_trip=False,
)

# 결과 확인
for date, tcn in results.items():
    hidden = (tcn['숨겨진환승횟수'] > 0).sum()
    print(f'{date}: {len(tcn):,} trips, 숨겨진환승 {hidden:,}건 ({hidden/len(tcn)*100:.1f}%)')

In [ ]:
dates = ['20250217', '20250218', '20250219', '20250220', '20250221', '20250222', '20250223']

results = process_multiple_dates(
    dates,
    base_path=r'C:\Folder\Research\0. DATA\tcd_2025_parquet',
    output_dir='../data/tcn_route',
    split_round_trip=False,
)

In [ ]:
# 결과 확인
for date, tcn in results.items():
    hidden = (tcn['숨겨진환승횟수'] > 0).sum()
    print(f'{date}: {len(tcn):,} trips, 숨겨진환승 {hidden:,}건 ({hidden/len(tcn)*100:.1f}%)')